In [ ]:
import pandas as pd
import re
import os
import matplotlib.pyplot as plt

In [ ]:
def extract_metadata_from_filename(filename: str) -> dict:
    # print(filename)
    pattern = re.compile(r"(?P<num_producers>\d+)p(?P<num_consumers>\d+)c(?P<num_topics>\d+)t\d+pc(?P<message_size>\d+)_0b\d+us\d+bm\-\d+[ms]-.*\.csv")
    match = pattern.search(filename)
    if not match:
        raise ValueError(f"Could not parse metadata from filename: {filename}")
    
    return {
        "num_producers": int(match.group("num_producers")),
        "num_consumers": int(match.group("num_consumers")),
        "num_topics": int(match.group("num_topics")),
        "message_size": int(match.group("message_size")),
    }

def extract_tech(filepath):
    return os.path.basename(os.path.dirname(filepath))

In [ ]:
def load_all_stats(root="../logs"):
    all_rows = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.endswith(".csv"):
                continue
            full_path = os.path.join(dirpath, fname)
            try:
                df = pd.read_csv(full_path, parse_dates=["timestamp"])
                df["timestamp"] = pd.to_datetime(df["timestamp"])
                df["elapsed_time"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds() * 1000
                # Extract metadata
                tech = os.path.basename(dirpath)
                meta = extract_metadata_from_filename(fname)
                meta["technology"] = tech
                for k, v in meta.items():
                    df[k] = v
                all_rows.append(df)
            except Exception as e:
                print(f"Failed to load {fname}: {e}")
    if not all_rows:
        raise RuntimeError("No stats files loaded.")
    return pd.concat(all_rows, ignore_index=True)


In [ ]:
stats_df = load_all_stats("../logs/quick_test_stats/")
stats_df = stats_df.drop(stats_df[stats_df.container_name.str.contains("broker")].index)
stats_df

In [ ]:
def filter_stats(df, *, technology=None, num_producers=1, num_consumers=2,
                 num_topics=4, message_size=None, role=None):
    filtered = df.copy()
    if technology:
        filtered = filtered[filtered["technology"] == technology]
    if num_producers:
        filtered = filtered[filtered["num_producers"] == num_producers]
    if num_consumers:
        filtered = filtered[filtered["num_consumers"] == num_consumers]
    if num_topics:
        filtered = filtered[filtered["num_topics"] == num_topics]
    if message_size:
        filtered = filtered[filtered["message_size"] == message_size]
    if role:
        filtered = filtered[filtered["container_name"].str.contains(f"-{role[0].upper()}")]
    return filtered


In [ ]:
def plot_metric_over_dimension(df, metric, title=None, x="elapsed_time", facet_by="container_name", agg_func="mean", unit_x = "ms", unit_y = None):
    plt.figure(figsize=(12, 6))
    if x in ["timestamp", "elapsed_time"]:
        # Time series mode — no aggregation
        for label, group in df.groupby(facet_by):
            plt.plot(group[x], group[metric], label=label)
        plt.xlabel(f"{x} ({unit_x})")
    else:
        # Dimensional mode — needs aggregation
        agg_df = (
            df.groupby([x, facet_by])[metric]
            .agg(agg_func)
            .reset_index()
            .sort_values(by=x)
        )

        for label, group in agg_df.groupby(facet_by):
            plt.plot(group[x], group[metric], marker="o", label=label)
        plt.xlabel("Time (ms)")

    plt.title(title or f"{metric} over time")
    plt.ylabel(metric.replace("_", " ").title() + f" ({unit_y})")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Slice only kafka data with 64B messages and 4 topics
df_kafka_64 = filter_stats(stats_df, technology="kafka", message_size=64, num_topics=4)

# Plot CPU usage for those containers
plot_metric_over_dimension(df_kafka_64, "cpu_usage_perc", "Kafka 64B CPU Usage (%)", unit_y="%")
plot_metric_over_dimension(df_kafka_64, "memory_usage", "Kafka 64B Memory Usage (B)", unit_y="B")
plot_metric_over_dimension(df_kafka_64, "network_tx", "Kafka 64B Bytes Sent", unit_y="B")
plot_metric_over_dimension(df_kafka_64, "network_rx", "Kafka 64B Bytes Received", unit_y="B")

In [ ]:
# Slice only zeromq_p2p data with 64B messages and 4 topics
df_zmq_64 = filter_stats(stats_df, technology="zeromq_p2p", message_size=64, num_topics=4)

# Plot CPU Usage (%) for those containers
plot_metric_over_dimension(df_zmq_64, "cpu_usage_perc", "ZeroMQP2P 64B CPU Usage (%)", unit_y="%")
plot_metric_over_dimension(df_zmq_64, "memory_usage", "ZeroMQP2P 64B Memory Usage (B)", unit_y="B")
plot_metric_over_dimension(df_zmq_64, "network_tx", "ZeroMQP2P 64B Bytes Sent", unit_y="B")
plot_metric_over_dimension(df_zmq_64, "network_rx", "ZeroMQP2P 64B Bytes Received", unit_y="B")

In [ ]:
# Slice only data with 64B messages and 4 topics
df_64 = filter_stats(stats_df, message_size=64, num_topics=4)

# Plot CPU Usage (%) for those containers
plot_metric_over_dimension(df_64, "cpu_usage_perc", "64B CPU Usage (%)", unit_y="%")
plot_metric_over_dimension(df_64, "memory_usage", "64B Memory Usage (B)", unit_y="B")
plot_metric_over_dimension(df_64, "network_tx", "64B Bytes Sent", unit_y="B")
plot_metric_over_dimension(df_64, "network_rx", "64B Bytes Received", unit_y="B")

In [ ]:
# Slice only publisher data with 64B messages and 4 topics
df_pub_64 = filter_stats(stats_df, role = "Publisher", message_size=64, num_topics=4)

# Plot CPU Usage (%) for those containers
plot_metric_over_dimension(df_pub_64, "cpu_usage_perc", "64B Publisher CPU Usage (%)", unit_y="%")
plot_metric_over_dimension(df_pub_64, "memory_usage", "64B Publisher Memory Usage (B)", unit_y="B")
plot_metric_over_dimension(df_pub_64, "network_tx", "64B Publisher Bytes Sent", unit_y="B")
plot_metric_over_dimension(df_pub_64, "network_rx", "64B Publisher Bytes Received", unit_y="B")

In [ ]:
# Slice only publisher data with 64B messages and 4 topics
df_con_64 = filter_stats(stats_df, role = "Consumer", message_size=64, num_topics=4)

# Plot CPU Usage (%) for those containers
plot_metric_over_dimension(df_con_64, "cpu_usage_perc", "64B Consumer CPU Usage (%)", unit_y="%")
plot_metric_over_dimension(df_con_64, "memory_usage", "64B Consumer Memory Usage (B)", unit_y="B")
plot_metric_over_dimension(df_con_64, "network_tx", "64B Consumer Bytes Sent", unit_y="B")
plot_metric_over_dimension(df_con_64, "network_rx", "64B Consumer Bytes Received", unit_y="B")

In [ ]:
# Slice only publisher data with 64B messages and 4 topics
df_zmq_pub = filter_stats(stats_df, technology="zeromq_p2p", role="Publi", num_topics=4)

# Plot CPU Usage (%) for those containers
plot_metric_over_dimension(df_zmq_pub, "cpu_usage_perc", "ZeroMQ Pub CPU Usage (%) /message size", facet_by="message_size", unit_y="%")
plot_metric_over_dimension(df_zmq_pub, "memory_usage", "ZeroMQ Pub Memory Usage /message size", facet_by="message_size", unit_y="B")
plot_metric_over_dimension(df_zmq_pub, "network_tx", "ZeroMQ Pub Bytes Sent /message size", facet_by="message_size", unit_y="B")
plot_metric_over_dimension(df_zmq_pub, "network_rx", "ZeroMQPub Bytes Received /message size", facet_by="message_size", unit_y="B")

In [ ]:
# Slice only publisher data with 64B messages and 4 topics
df_kafka_pub = filter_stats(stats_df, technology="kafka", role="Publi", num_topics=4)

# Plot CPU Usage (%) for those containers
plot_metric_over_dimension(df_kafka_pub, "cpu_usage_perc", "Kafka Pub CPU Usage (%) /message size", facet_by="message_size", unit_y="%")
plot_metric_over_dimension(df_kafka_pub, "memory_usage", "Kafka Pub Memory Usage /message size", facet_by="message_size", unit_y="B")
plot_metric_over_dimension(df_kafka_pub, "network_tx", "Kafka Pub Bytes Sent /message size", facet_by="message_size", unit_y="B")
plot_metric_over_dimension(df_kafka_pub, "network_rx", "Kafka Bytes Received /message size", facet_by="message_size", unit_y="B")

In [ ]:
def plot_metric_over_dimension(df, metric, title=None, x="elapsed_time", facet_by="container_name", agg_func="mean", unit_x = "ms", unit_y = None):
    plt.figure(figsize=(12, 6))
    if x in ["timestamp", "elapsed_time"]:
        # Time series mode — no aggregation
        for label, group in df.groupby(facet_by):
            plt.plot(group[x], group[metric], label=label)
    else:
        # Dimensional mode — needs aggregation
        agg_df = (
            df.groupby([x, facet_by])[metric]
            .agg(agg_func)
            .reset_index()
            .sort_values(by=x)
        )

        for label, group in agg_df.groupby(facet_by):
            plt.plot(group[x], group[metric], marker="o", label=label)

    plt.title(title or f"{metric} over time")
    plt.xlabel(f"{x} ({unit_x})")
    plt.ylabel(metric.replace("_", " ").title() + f" ({unit_y})")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# plot_metric_over_dimension(df_kafka_pub, "memory_usage", "Kafka Pub CPU Usage (%) vs msg size", x="message_size", unit_x="B", unit_y="%")

df_kafka = filter_stats(stats_df, technology="kafka", num_topics=4)

plot_metric_over_dimension(df_kafka, "cpu_usage_perc", "Kafka Pub CPU Usage (%) /message size",  x="message_size", unit_x="B", unit_y="%")
plot_metric_over_dimension(df_kafka, "memory_usage", "Kafka Pub Memory Usage /message size",  x="message_size", unit_x="B", unit_y="%")
plot_metric_over_dimension(df_kafka, "network_tx", "Kafka Pub Bytes Sent /message size",  x="message_size", unit_x="B", unit_y="%")
plot_metric_over_dimension(df_kafka, "network_rx", "Kafka Bytes Received /message size",  x="message_size", unit_x="B", unit_y="%")